In [1]:
import os
import shutil
import kagglehub
import re
import pandas as pd
import numpy as np

d:\US\Data mining\group_proj\Group_12_Charm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Tải dataset về cache
cache_path = kagglehub.dataset_download("eliasdabbas/web-server-access-logs")

target_dir = os.path.abspath(os.path.join("..", "..", "data", "application", "web_server_access_logs"))
os.makedirs(target_dir, exist_ok=True)

for file_name in os.listdir(cache_path):
    if file_name.endswith(".log"):
        src_file = os.path.join(cache_path, file_name)
        dst_file = os.path.join(target_dir, file_name)
    
    # Nếu file đích đã tồn tại thì xóa trước để tránh lỗi ghi đè của shutil.move
    if os.path.exists(dst_file):
        os.remove(dst_file) if os.path.isfile(dst_file) else shutil.rmtree(dst_file)
        
    shutil.move(src_file, dst_file)

print("Đã di chuyển toàn bộ file vào:", target_dir)
print("Danh sách file:", os.listdir(target_dir))

Đã di chuyển toàn bộ file vào: d:\US\Data mining\group_proj\Group_12_Charm\data\application\web_server_access_logs
Danh sách file: ['access.log', 'client_hostname.csv']


In [3]:
access_logs_path = os.path.join(target_dir, "access.log")
i = 0
lines = []
with open(access_logs_path, "r", encoding="utf-8") as f:
    for line in f:
        lines.append(line)

In [4]:
print(len(lines))

10365152


In [5]:
REGEX_PATTERN = re.compile(
    # IP
    r'(?P<ip>\S+)\s+\S+\s+\S+\s+'
    # date time
    r'\[(?P<time>[^\]]+)\]\s+'
    # request
    r'"(?P<request>[^"]*)"\s+'
    # status & size
    r'(?P<status>\d{3})\s+(?P<size>\S+)\s+'
    # referer
    r'"(?P<referrer>[^"]*)"\s+'
    # agent
    r'"(?P<agent>[^"]*)"\s+'
    # extra
    r'"(?P<extra>[^"]*)"'
)


In [6]:
results = []
j = 0
for line in lines:
    match = REGEX_PATTERN.match(line)
    
    if match:
        results.append(match.groupdict())
    else:
        print("Không match:", line)
    j+=1
    if (j+1) % 1_000_000 == 0 or j == 0:
        print(f"LINE {j+1}: DONE")
    
# Hiển thị kết quả
j = 0
for r in results:
    print(r)
    if j == 5:
        break
    j+=1

LINE 1000000: DONE
LINE 2000000: DONE
LINE 3000000: DONE
LINE 4000000: DONE
LINE 5000000: DONE
LINE 6000000: DONE
LINE 7000000: DONE
LINE 8000000: DONE
LINE 9000000: DONE
LINE 10000000: DONE
{'ip': '54.36.149.41', 'time': '22/Jan/2019:03:56:14 +0330', 'request': 'GET /filter/27|13%20%D9%85%DA%AF%D8%A7%D9%BE%DB%8C%DA%A9%D8%B3%D9%84,27|%DA%A9%D9%85%D8%AA%D8%B1%20%D8%A7%D8%B2%205%20%D9%85%DA%AF%D8%A7%D9%BE%DB%8C%DA%A9%D8%B3%D9%84,p53 HTTP/1.1', 'status': '200', 'size': '30577', 'referrer': '-', 'agent': 'Mozilla/5.0 (compatible; AhrefsBot/6.1; +http://ahrefs.com/robot/)', 'extra': '-'}
{'ip': '31.56.96.51', 'time': '22/Jan/2019:03:56:16 +0330', 'request': 'GET /image/60844/productModel/200x200 HTTP/1.1', 'status': '200', 'size': '5667', 'referrer': 'https://www.zanbil.ir/m/filter/b113', 'agent': 'Mozilla/5.0 (Linux; Android 6.0; ALE-L21 Build/HuaweiALE-L21) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/66.0.3359.158 Mobile Safari/537.36', 'extra': '-'}
{'ip': '31.56.96.51', 'time': '22/Ja

In [7]:
df = pd.DataFrame(results)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10365152 entries, 0 to 10365151
Data columns (total 8 columns):
 #   Column    Dtype
---  ------    -----
 0   ip        str  
 1   time      str  
 2   request   str  
 3   status    str  
 4   size      str  
 5   referrer  str  
 6   agent     str  
 7   extra     str  
dtypes: str(8)
memory usage: 632.6 MB


In [9]:
df[['method', 'raw_url']] = (
    df['request'].str.split(' ', n=2, expand=True).iloc[:, :2]
)

REGEX_MODULE = re.compile(
    r'^(?:/m)?/(?P<module>[^/?]+)'
)

# Trích xuất ra Module
df['module'] = (
    df['raw_url']
    .str.extract(REGEX_MODULE, expand=False)
    .str.upper()
    .fillna("HOME")
        )
# Check Module đó có phải là file static không và thay đổi
is_file_with_ext = df['module'].str.contains(r'\.', regex=True, na=False)
df.loc[is_file_with_ext & (df['module'] != 'ROBOTS.TXT'), 'module'] = 'STATIC'

# Sửa các lỗi dính chuỗi 'SEARCH%...' (Item 513) và 'AHREFS_...' (Item 350, 351)
mod_str = df['module'].astype(str)
df.loc[mod_str.str.startswith('SEARCH%'), 'module'] = 'SEARCH'
df.loc[mod_str.str.startswith('AHREFS_'), 'module'] = 'ROBOTS'

# Gộp các module liên quan lại
df['module'] = df['module'].replace({
    # Trang chủ
    'M': 'HOME',
    'INDEX': 'HOME',
    # Hình ảnh
    'IMAGES': 'IMAGE',
    'PRODUCTIMAGE': 'IMAGE',
    'SPECIALSALESLIDE': 'IMAGE',
    'SLIDE': 'IMAGE',
    # Tài nguyên tĩnh (Static)
    'PLUGINS': 'STATIC',
    'JS': 'STATIC',
    'PUBLIC': 'STATIC',
    'MEDIA': 'STATIC',
    'SKINS': 'STATIC',
    'FILES': 'STATIC',
    'APPLE-APP-SITE-ASSOCIATION': 'STATIC',
    'CROSS-PLATFORM-APP-IDENTIFIERS': 'STATIC',
    'FA5': 'STATIC',
    'SIGNATUREPAD': 'STATIC',
    'SIMPLECAPTCHA': 'STATIC',
    'SIDEBAR': 'STATIC',
    # Sản phẩm & Thuộc tính sản phẩm
    'PRODUCTMODEL': 'PRODUCT',
    'PRODUCTS': 'PRODUCT',
    'PRODUCERPRODUCTMODEL': 'PRODUCT',
    'PRODUCINGPRODUCT': 'PRODUCT',
    'UPDATEVARIATION': 'PRODUCT_ATTR',
    'VARIATIONGROUP': 'PRODUCT_ATTR',
    'PRODUCTTYPE': 'PRODUCT_ATTR',
    'ADDEDVALUE': 'PRODUCT_ATTR',
    'PRODUCER': 'PRODUCT_ATTR',
    'SUPPLIERS': 'PRODUCT_ATTR',
    'CUSTOMERREVIEW': 'PRODUCT_ATTR',
    # Nội dung, Tìm kiếm & Sự kiện
    'ARTICLELIST': 'ARTICLE',
    'AJAXSEARCH': 'SEARCH',
    'SEARCHAUTOCOMPLETE': 'SEARCH',
    'PREPARESEARCH': 'SEARCH',
    'CULTUREEVENT': 'EVENT',
    'ABOUTUS': 'CONTENT',
    'CONTACTUS': 'CONTENT',
    'FAQ': 'CONTENT',
    'TERMSANDCONDITIONS': 'CONTENT',
    'SHOPPINGRULES': 'CONTENT',
    'SHOPPINGSTEPS': 'CONTENT',
    'TRUST': 'CONTENT',
    'ENAMAD': 'CONTENT',
    # Đơn hàng, Giao hàng, Giá & Khuyến mãi
    'ORDERADMINISTRATION': 'ORDER',
    'ORDERCOLLECTION': 'ORDER',
    'PAYMENTREQUEST': 'PAYMENT',
    'PAYMENTMETHODS': 'PAYMENT',
    'PAYMENTANDDELIVERY': 'PAYMENT',
    'DELIVERYTIPS': 'DELIVERY',
    'DELIVERYPRICE': 'DELIVERY',
    'DELIVERYMETHOD': 'DELIVERY',
    'PRICEGUARANTEE': 'PRICE',
    'OTHERPRICELINK': 'PRICE',
    'PRICINGLINK': 'PRICE',
    'GUARANTEE': 'PRICE',
    'GUARANTEEDAYS': 'PRICE',
    'MONEYBACKCONDITIONS': 'PRICE',
    'DISCOUNTNEW': 'DISCOUNTLABEL',
    # Khách hàng & Xác thực
    'USER': 'CUSTOMER',
    'REGISTER': 'CUSTOMER',
    'WISHLIST': 'CUSTOMER',
    'MESSAGE': 'CUSTOMER',
    'J_SPRING_SECURITY_CHECK': 'LOGIN',
    'J_SPRING_SECURITY_LOGOUT': 'LOGOUT',
    # Hệ thống, Bot & Báo lỗi
    'ROBOTS.TXT': 'ROBOTS',
    'SITEMAP': 'ROBOTS',
    'RSS': 'ROBOTS',
    'ALEXAGOOLEANALITIC': 'SITE',
    'TRACK': 'SITE',
    'GO': 'SITE',
    'APP': 'SITE',
    'GRAILS-ERRORHANDLER': 'ERRORS',
    'RAPIDGRAILS': 'ADMIN',
})

# File Robots chuyển riêng
df['module'] = df['module'].replace({'ROBOTS.TXT': 'ROBOTS'})

# Check Method hợp lệ
VALID_METHODS = ['GET', 'POST', 'HEAD', 'OPTIONS', 'PUT', 'DELETE']
valid_method_count = len(df['method'].isin(VALID_METHODS))

valid_mask = (
    df['method'].isin(VALID_METHODS)
    & df['raw_url'].notna()
    & df['raw_url'].str.startswith('/', na=False)
)

df.loc[~valid_mask, 'method'] = 'BAD'
df.loc[~valid_mask, 'module'] = 'MALFORMED_PROBE'

df.drop(columns=['request', 'raw_url', 'extra'], inplace=True, errors='ignore')

C:\Users\Quan\AppData\Local\Temp\ipykernel_11136\2402993015.py:125: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=['request', 'raw_url', 'extra'], inplace=True, errors='ignore')


In [10]:
# Gom nhóm 3 chiến dịch Tấn công / Quét lỗ hổng (DB, CMS, Exploit)
m_col = df['module']
cond_scan_db = m_col.str.contains(
    'PHP|PMA|MYADMIN|MYSQL|SQL|DB|XAMPP|PMD|SHAADMIN|CPADMIN',
    regex=True,
    na=False,
)
cond_scan_cms = m_col.str.contains(
    'WP|WORDPRESS|MAGENTO|MAGE|TYPO3|CMS|CLAROLINE|NETCAT|ADMINISTRATOR|BJADMIN|LOJA|SHOP|STORE|MARKET|MAG',
    regex=True,
    na=False,
)
cond_scan_vuln = m_col.isin({
    'GPONFORM',
    'HNAP1',
    '_VTI_BIN',
    'MSOFFICE',
    'CGI-BIN',
    'CURRENT_CONFIG',
    'SERVER-STATUS',
    'WEBDAV',
    'BACKUPS',
    'BACKUP',
    'CONFIG',
    'PWD',
    'CACTI',
    'INDY',
    'SCRIPTS',
    'UPLOAD',
    'TEMP',
    'SYSTEM',
    'INCLUDES',
    'ENGINE',
    'MODULES',
    'MANAGER',
    'FRONTEND',
    'WEB',
    'WWW',
    'TOOLS',
    'EXAMPLES',
    'HEALTH',
    'IBM',
    'TEST',
    'OLD',
    'NEW',
    'DEV',
    'DEMO',
    'DEFAULT',
    'PROGRAM',
    'STORAGE',
    'STOCK',
    'ADVS',
    'ETM',
})

df['module'] = np.select(
    [cond_scan_db, cond_scan_cms, cond_scan_vuln],
    ['SCAN_DB_ADMIN', 'SCAN_CMS_ADMIN', 'SCAN_EXPLOIT_VULN'],
    default=m_col,
)

In [12]:
# Khóa Whitelist: Mọi chuỗi rác ngẫu nhiên (FDSRWE, QNFXCJQR, X, V, S...) gom vào UNKNOWN_SCAN_PATH
CLEAN_WHITELIST = {
    'HOME',
    'PRODUCT',
    'PRODUCT_ATTR',
    'FILTER',
    'AJAXFILTER',
    'BROWSE',
    'SEARCH',
    'IMAGE',
    'STATIC',
    'SETTINGS',
    'DISCOUNTLABEL',
    'BASKET',
    'ORDER',
    'PAYMENT',
    'DELIVERY',
    'PRICE',
    'CUSTOMER',
    'LOGIN',
    'LOGOUT',
    'COMPARISON',
    'ARTICLE',
    'BLOG',
    'EVENT',
    'CONTENT',
    'PROVINCE',
    'SITE',
    'ROBOTS',
    'ADMIN',
    'ERRORS',
    'MALFORMED_PROBE',
    'INVALID_URI_TYPO',
    'SCAN_DB_ADMIN',
    'SCAN_CMS_ADMIN',
    'SCAN_EXPLOIT_VULN',
    'UNKNOWN_SCAN_PATH',
}
df.loc[~df['module'].isin(CLEAN_WHITELIST), 'module'] = 'UNKNOWN_SCAN_PATH'

In [13]:
df.head()

,ip,time,status,size,referrer,agent,method,module
0,54.36.149.41,22/Jan/2019:03:56:14 +0330,200,30577,-,Mozilla/5.0 (compatible; AhrefsBot/6.1; +http:...,GET,FILTER
1,31.56.96.51,22/Jan/2019:03:56:16 +0330,200,5667,https://www.zanbil.ir/m/filter/b113,Mozilla/5.0 (Linux; Android 6.0; ALE-L21 Build...,GET,SCAN_CMS_ADMIN
2,31.56.96.51,22/Jan/2019:03:56:16 +0330,200,5379,https://www.zanbil.ir/m/filter/b113,Mozilla/5.0 (Linux; Android 6.0; ALE-L21 Build...,GET,SCAN_CMS_ADMIN
3,40.77.167.129,22/Jan/2019:03:56:17 +0330,200,1696,-,Mozilla/5.0 (compatible; bingbot/2.0; +http://...,GET,SCAN_CMS_ADMIN
4,91.99.72.15,22/Jan/2019:03:56:17 +0330,200,41483,-,Mozilla/5.0 (Windows NT 6.2; Win64; x64; rv:16...,GET,PRODUCT


In [14]:
df['module'].unique()

<StringArray>
[           'FILTER',    'SCAN_CMS_ADMIN',           'PRODUCT',
          'SETTINGS',            'BROWSE',           'ARTICLE',
              'HOME',            'STATIC',      'PRODUCT_ATTR',
        'AJAXFILTER',              'BLOG',              'SITE',
            'ROBOTS',             'LOGIN',            'SEARCH',
     'DISCOUNTLABEL',        'COMPARISON',             'EVENT',
          'CUSTOMER',             'PRICE',            'BASKET',
          'PROVINCE',           'CONTENT', 'UNKNOWN_SCAN_PATH',
     'SCAN_DB_ADMIN',             'ORDER',            'ERRORS',
             'ADMIN',          'DELIVERY',   'MALFORMED_PROBE',
           'PAYMENT',            'LOGOUT', 'SCAN_EXPLOIT_VULN']
Length: 33, dtype: str

In [15]:
print(df['module'].dropna().unique().tolist())

['FILTER', 'SCAN_CMS_ADMIN', 'PRODUCT', 'SETTINGS', 'BROWSE', 'ARTICLE', 'HOME', 'STATIC', 'PRODUCT_ATTR', 'AJAXFILTER', 'BLOG', 'SITE', 'ROBOTS', 'LOGIN', 'SEARCH', 'DISCOUNTLABEL', 'COMPARISON', 'EVENT', 'CUSTOMER', 'PRICE', 'BASKET', 'PROVINCE', 'CONTENT', 'UNKNOWN_SCAN_PATH', 'SCAN_DB_ADMIN', 'ORDER', 'ERRORS', 'ADMIN', 'DELIVERY', 'MALFORMED_PROBE', 'PAYMENT', 'LOGOUT', 'SCAN_EXPLOIT_VULN']


In [16]:
# Group lại
df['time'] = pd.to_datetime(df['time'], format='%d/%b/%Y:%H:%M:%S %z', errors='coerce')
df['window_5m'] = df['time'].dt.floor('5min')

df['event_action'] = df['method'] + '_' + df['module'] + '_' + df['status']

agent_lower = df['agent'].str.lower().fillna('-')

cond_list = [
    # Điều kiện 1: User-Agent bị bỏ trống hoặc là dấu "-" (thường là script/tool quét lỗi)
    (df['agent'] == '-') | (df['agent'] == ''),
    # Điều kiện 2: Chứa các từ khóa của Bot, Crawler hoặc công cụ tự động
    agent_lower.str.contains(
        'bot|crawl|spider|ahrefs|bing|slurp|yandex|baidu|python|curl|wget|headless|scrapy',
        regex=True,
        na=False,
    ),
    # Điều kiện 3: Chứa từ khóa của thiết bị di động
    agent_lower.str.contains(
        'android|iphone|ipad|mobile', regex=True, na=False
    ),
]

choice_list = ['UA_EMPTY', 'UA_KNOWN_BOT', 'UA_MOBILE']
df['event_agent'] = np.select(cond_list, choice_list, default='UA_DESKTOP')

#  Tạo nhãn phân loại nguồn dẫn (Referrer)
df['event_ref'] = np.where(df['referrer'] == '-', 'REF_DIRECT', 'REF_LINKED')

df.drop(
    columns=[
        'request',
        'raw_url',
        'extra',
        'time',
        'agent',
        'referrer',
        'method',
        'size',
    ],
    inplace=True,
    errors='ignore',
)

df.head()

C:\Users\Quan\AppData\Local\Temp\ipykernel_11136\210755558.py:30: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(


,ip,status,module,window_5m,event_action,event_agent,event_ref
0,54.36.149.41,200,FILTER,2019-01-22 03:55:00+03:30,GET_FILTER_200,UA_KNOWN_BOT,REF_DIRECT
1,31.56.96.51,200,SCAN_CMS_ADMIN,2019-01-22 03:55:00+03:30,GET_SCAN_CMS_ADMIN_200,UA_MOBILE,REF_LINKED
2,31.56.96.51,200,SCAN_CMS_ADMIN,2019-01-22 03:55:00+03:30,GET_SCAN_CMS_ADMIN_200,UA_MOBILE,REF_LINKED
3,40.77.167.129,200,SCAN_CMS_ADMIN,2019-01-22 03:55:00+03:30,GET_SCAN_CMS_ADMIN_200,UA_KNOWN_BOT,REF_DIRECT
4,91.99.72.15,200,PRODUCT,2019-01-22 03:55:00+03:30,GET_PRODUCT_200,UA_DESKTOP,REF_DIRECT


In [17]:
# Gom Nhóm
CONTENT_MODULES = {
    'PRODUCT',
    'PRODUCT_ATTR',
    'FILTER',
    'BROWSE',
    'ARTICLE',
    'BLOG',
    'SEARCH',
    'EVENT',
    'CONTENT',
    'COMPARISON',
    'HOME',
}
ASSET_MODULES = {'IMAGE', 'STATIC', 'SETTINGS', 'DISCOUNTLABEL'}

df['is_content'] = df['module'].isin(CONTENT_MODULES)
df['is_asset'] = df['module'].isin(ASSET_MODULES)
df['is_4xx'] = df['status'].str.startswith('4', na=False)

# Gom nhóm theo (ip, window_5m)
sessions_df = (
    df.groupby(['ip', 'window_5m'], sort=False)
    .agg(
        actions=('event_action', set),
        agents=('event_agent', set),
        refs=('event_ref', set),
        has_content=('is_content', 'any'),
        has_asset=('is_asset', 'any'),
        err_4xx_count=('is_4xx', 'sum'),
        req_count=('status', 'count'),
    )
    .reset_index(drop=True)
)

# Đóng lại thành transaction
transactions = []

for row in sessions_df.itertuples(index=False):
    
    basket = set()
    basket.update(row.actions)
    basket.update(row.agents)
    basket.update(row.refs)
    
    # Cường độ truy cập
    if row.req_count == 1:
        basket.add('RATE_SINGLE')
    elif row.req_count <= 20:
        basket.add('RATE_NORMAL')
    elif row.req_count <= 100:
        basket.add('RATE_HIGH')
    else:
        basket.add('RATE_EXTREME_FLOOD')
        
    # Tải tài nguyên từ Bot Scraper
    if row.has_content and not row.has_asset:
        basket.add('BEHAVIOR_NO_ASSET_SCRAPING')
    elif row.has_content and row.has_asset:
        basket.add('BEHAVIOR_NORMAL_ASSET_LOAD')
        
    # Quét lỗ hỏng error
    if row.err_4xx_count >= 3:
        basket.add('BEHAVIOR_4XX_SCANNING_BURST')

    transactions.append(basket)

print(f'Tổng số giỏ giao dịch (5-min IP sessions): {len(transactions)}')
print('Thử kiểm tra 5 giỏ đầu tiên:')
for i, t in enumerate(transactions[:5]):
  print(f'Giỏ {i+1}: {sorted(list(t))}')

Tổng số giỏ giao dịch (5-min IP sessions): 520428
Thử kiểm tra 5 giỏ đầu tiên:
Giỏ 1: ['BEHAVIOR_NO_ASSET_SCRAPING', 'GET_FILTER_200', 'RATE_SINGLE', 'REF_DIRECT', 'UA_KNOWN_BOT']
Giỏ 2: ['BEHAVIOR_NORMAL_ASSET_LOAD', 'GET_BROWSE_200', 'GET_FILTER_200', 'GET_PRODUCT_200', 'GET_SCAN_CMS_ADMIN_200', 'GET_SETTINGS_200', 'GET_STATIC_200', 'POST_COMPARISON_200', 'RATE_HIGH', 'REF_LINKED', 'UA_MOBILE']
Giỏ 3: ['GET_SCAN_CMS_ADMIN_200', 'RATE_NORMAL', 'REF_DIRECT', 'UA_KNOWN_BOT']
Giỏ 4: ['BEHAVIOR_NO_ASSET_SCRAPING', 'GET_PRODUCT_200', 'RATE_HIGH', 'REF_DIRECT', 'UA_DESKTOP']
Giỏ 5: ['BEHAVIOR_4XX_SCANNING_BURST', 'BEHAVIOR_NO_ASSET_SCRAPING', 'GET_ARTICLE_200', 'GET_BLOG_200', 'GET_BROWSE_200', 'GET_BROWSE_302', 'GET_FILTER_200', 'GET_FILTER_302', 'GET_HOME_200', 'GET_PRODUCT_200', 'GET_PRODUCT_302', 'GET_PRODUCT_404', 'GET_SEARCH_404', 'RATE_EXTREME_FLOOD', 'REF_DIRECT', 'UA_KNOWN_BOT']


In [ ]:
item_to_id = {}
next_id = 1

output_dir = os.path.join(target_dir, 'pre_process')
os.makedirs(output_dir, exist_ok=True)

map_path = os.path.join(output_dir, 'mapping.csv')
txt_path = os.path.join(output_dir, 'transaction.txt')

# Ghi file giao dịch cho CHARM (chỉ dùng item_to_id)
with open(txt_path, 'w', encoding='utf-8') as f_out:
  for basket in transactions:
    encoded_line = []
    for item in basket:
      if item not in item_to_id:
        item_to_id[item] = next_id
        next_id += 1
      encoded_line.append(item_to_id[item])

    encoded_line.sort()
    f_out.write(' '.join(map(str, encoded_line)) + '\n')

# Khi viết ra file mapping: Đảo trực tiếp vị trí (item, id) -> (id, item)
mapping_df = pd.DataFrame(
    [(item_id, item_name) for item_name, item_id in item_to_id.items()],
    columns=['ItemID', 'EventName'],
)
mapping_df.to_csv(map_path, index=False, encoding='utf-8')

print(f'DONE! Tổng số giỏ: {len(transactions)} | Tổng số Items sạch: {len(item_to_id)}')
print(f'Đã lưu tại:\n - {txt_path}\n - {map_path}')

DONE! Tổng số giỏ: 520428 | Tổng số Items sạch: 324
Đã lưu tại:
 - d:\US\Data mining\group_proj\Group_12_Charm\data\application\web_server_access_logs\pre_process\transaction.txt
 - d:\US\Data mining\group_proj\Group_12_Charm\data\application\web_server_access_logs\pre_process\mapping.csv
